In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

Load data

In [4]:
loader=PyPDFLoader("../data/medical_report.pdf")
docs=loader.load()

In [5]:
len(docs)

9

split data into chunks

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [7]:
len(splitted_docs)

26

In [8]:
embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3205.66it/s]


In [26]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF documents, and these documents have details about medical reports.
    """
    print("Tool called: ",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""

    for doc in docs:
        context+=doc.page_content+ "\n\n"

    return context

In [ ]:
# retriever_tool.invoke("patient Name")

'MD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\nDr Kiran Bhargava Pathak\nMD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\n*474764803*\n.\nPage 7 of 9\n\nReport Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status Final\n10/7/2025  6:31:50PM\n:Collected at            :Processed at             BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL-NATIONAL REFERENCE LAB\nNational Reference laboratory, Block E, \nSector 18, Rohini, New Delhi -110085\nTest Report      \nTest Name Results Units Bio. Ref. Interval\nHLA - B27\n(Flow Cytometry)\nHLA-B27, Disease Association   Negative\nInterpretation\n ------------------------------------------------------------\n| RESULT         |         REMARKS         

In [14]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [20]:
System_prompt="""
    You are a helpful assistant that answers questions using retrieved context.
ALWAYS use the retriever_tool tool for questions requiring external knowledge.
"""

In [27]:
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_prompt
)

In [28]:
query="what is the name of the patient, and what is the name og doctor"
response=agent.invoke({"messages":[{"role":"user","content":query}]})


Tool called:  patient name doctor name medical report


In [29]:
result=response["messages"][-1].content

In [30]:
print(result)

**Patient:** Ms. Nikita Chudhary  
**Doctor:** Dr. Nitin Nahar  

These names are listed in the report header – the patient is identified as “Ms. Nikita Chudhary” and the reporting physician is noted as “DR NITIN NAHAR.”
